In [1]:
""" import fiftyone as fo 
import fiftyone.brain as fob
from fiftyone import ViewField as F 
import os
"""

' import fiftyone as fo \nimport fiftyone.brain as fob\nfrom fiftyone import ViewField as F \nimport os '

In [ ]:
from anomalib.data import Folder
from anomalib.models import Padim, Patchcore
from anomalib.engine import Engine
import torch

def train_socket_inspector():
    """ model = Patchcore(
        backbone="resnet18",  # Better feature extraction
        layers=["layer2", "layer3"],  # Multi-scale features
    )
    
    # More training configs
    datamodule = Folder(
        name="socket_pins",
        root="./datasets/socket_pins", 
        normal_dir="train/good",
    ) """

    model = Padim(backbone="resnet18")
    datamodule = Folder(
        name="socket_pins",
        root="./datasets/socket_pins", 
        normal_dir="train/good",
    )
    
    # 3. Training Engine
    # 'image_metrics' gives you an AUROC score to judge accuracy
    engine = Engine()
    
    # 4. Fit the model
    print("Starting training...")
    engine.fit(datamodule=datamodule, model=model)
    
    # 5. Test (Optional, if you have a test set)
    engine.test(datamodule=datamodule, model=model)
    print("Training complete. Model saved.")

    engine.export(
        model=model,
        export_type="torch", # or "openvino" for Intel optimization
        ckpt_path="results/Padim/socket_pins/latest/weights/lightning/model.ckpt",
    )
    print("Model exported to .pt format successfully.")

if __name__ == "__main__":
    train_socket_inspector()

In [ ]:
from anomalib.deploy import TorchInferencer
import cv2
import matplotlib.pyplot as plt
import dotenv
import os

dotenv.load_dotenv()

def inspect_pin(model_path, pin_image_path):
    # 1. Load Model
    inferencer = OpenVINOInferencer(path=model_path, device="cpu")
    
    # 2. Predict
    predictions = inferencer.predict(image=pin_image_path)
    
    # --- FIX 1: HANDLE HEATMAP SHAPE ---
    anom_map = predictions.anomaly_map
    if hasattr(anom_map, "cpu"): 
        anom_map = anom_map.cpu().numpy()
    anom_map = anom_map.squeeze() # Removes (1, 256, 256) -> (256, 256)

    # --- FIX 2: HANDLE SCORE FORMATTING ---
    # Convert Tensor to float using .item()
    score = predictions.pred_score.item() 

    # 3. Visualize
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    
    # Original Image
    ax[0].imshow(cv2.cvtColor(cv2.imread(pin_image_path), cv2.COLOR_BGR2RGB))
    ax[0].set_title("Input Pin")
    ax[0].axis('off')
    
    # Anomaly Heatmap
    ax[1].imshow(anom_map, cmap='inferno') 
    ax[1].set_title(f"Score: {score:.2f}") # Now valid because 'score' is a float
    ax[1].axis('off')
    
    plt.tight_layout()
    plt.show()

img_path = "./datasets/socket_pins/test/defect/"
for img in os.listdir(img_path):
    inspect_pin("", f"{img_path}{img}")